# Stage 2: Dataset Integration & Phase Analysis
**Project:** KAN-Driven Phase-Spectrum Analysis for Deepfake Detection

In [ ]:
!pip install kagglehub -q

In [ ]:
import numpy as np, pandas as pd, cv2, os, glob
from typing import Dict, List, Tuple, Optional
from scipy import stats
import matplotlib.pyplot as plt, seaborn as sns
from tqdm.notebook import tqdm
import kagglehub

%matplotlib inline
plt.rcParams['figure.dpi'] = 120; sns.set_style('whitegrid')

INPUT_DIR = kagglehub.dataset_download('awsaf49/artifact-dataset')
OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else './output'
CACHE_DIR = os.path.join(OUTPUT_DIR, 'phase_cache')
os.makedirs(CACHE_DIR, exist_ok=True)

print(f'Dataset: {INPUT_DIR}')
print(f'Cache  : {CACHE_DIR}')
for item in sorted(os.listdir(INPUT_DIR))[:15]:
    f = os.path.join(INPUT_DIR, item)
    print(f'  {"[DIR]" if os.path.isdir(f) else "[FILE]"} {item}')

In [ ]:
# Cell 2: Dataset Discovery
class DatasetExplorer:
    def __init__(self, root): self.root = root; self.metadata_df = None
    def load_metadata(self):
        meta_files = glob.glob(os.path.join(self.root, '**', 'metadata.csv'), recursive=True)
        print(f'Found {len(meta_files)} metadata.csv files')
        all_dfs = []
        for mf in meta_files:
            gen_dir = os.path.dirname(mf)
            gen_name = os.path.basename(gen_dir)
            df = pd.read_csv(mf)
            df['generator'] = gen_name
            df['image_path'] = df['image_path'].apply(lambda p: os.path.join(gen_dir, p) if not os.path.isabs(p) else p)
            all_dfs.append(df)
            print(f'  {len(df):>6d} from [{gen_name}]')
        if not all_dfs:
            return self._fallback()
        self.metadata_df = pd.concat(all_dfs, ignore_index=True)
        if 'target' in self.metadata_df.columns:
            self.metadata_df['target'] = self.metadata_df['target'].astype(int)
        return self.metadata_df
    def _fallback(self):
        exts = {'.png','.jpg','.jpeg','.bmp','.webp'}
        recs = []
        for root, _, files in os.walk(self.root):
            for f in files:
                if os.path.splitext(f)[1].lower() in exts:
                    full = os.path.join(root, f)
                    parts = os.path.relpath(root, self.root).lower().split(os.sep)
                    is_real = 'real' in parts
                    gen = 'real' if is_real else (parts[-1] if parts[-1] != '.' else 'unknown')
                    recs.append({'image_path':full,'target':0 if is_real else 1,'generator':gen})
        self.metadata_df = pd.DataFrame(recs)
        print(f'Fallback scan: {len(self.metadata_df)} images')
        return self.metadata_df
    def summary(self):
        df = self.metadata_df
        print(f'Total: {len(df):,}  Real: {(df["target"]==0).sum():,}  Fake: {(df["target"]==1).sum():,}')
        for gen, cnt in df['generator'].value_counts().items():
            lbl = 'REAL' if df[df['generator']==gen]['target'].mode().iloc[0]==0 else 'FAKE'
            print(f'  {gen:30s}: {cnt:>6,} [{lbl}]')

explorer = DatasetExplorer(INPUT_DIR)
metadata_df = explorer.load_metadata()
explorer.summary()

In [ ]:
# Cell 3: Batch Phase Extraction
class PhaseSpectrumExtractor:
    _LR,_LG,_LB = 0.2989,0.5870,0.1140
    def __init__(self, size=(256,256)): self.size = size
    def process_single(self, path):
        try:
            bgr = cv2.imread(path, cv2.IMREAD_COLOR)
            if bgr is None: return None
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float64)
            gray = self._LR*rgb[:,:,0]+self._LG*rgb[:,:,1]+self._LB*rgb[:,:,2]
            gray = cv2.resize(gray, (self.size[1],self.size[0]), interpolation=cv2.INTER_CUBIC)
            fft = np.fft.fftshift(np.fft.fft2(gray))
            phase = np.angle(fft)
            mn,mx = phase.min(),phase.max()
            return (phase-mn)/(mx-mn) if mx>mn else np.zeros(self.size)
        except: return None

class BatchPhaseExtractor:
    def __init__(self, cache_dir): self.cache_dir=cache_dir; self.ext=PhaseSpectrumExtractor(); os.makedirs(cache_dir, exist_ok=True)
    def extract(self, df, n_per_gen=50, use_cache=True):
        cf = os.path.join(self.cache_dir, f'phase_maps_n{n_per_gen}.npy')
        if use_cache and os.path.exists(cf):
            print(f'Loading cache: {cf}')
            r = np.load(cf, allow_pickle=True).item()
            for g,m in r.items(): print(f'  {g:30s}: {len(m)} maps')
            return r
        results = {}
        for gen in sorted(df['generator'].unique()):
            gdf = df[df['generator']==gen]
            if n_per_gen>0 and len(gdf)>n_per_gen: gdf=gdf.sample(n=n_per_gen, random_state=42)
            maps, fail = [], 0
            for p in tqdm(gdf['image_path'].values, desc=f'{gen[:25]:25s}'):
                m = self.ext.process_single(p)
                if m is not None: maps.append(m)
                else: fail+=1
            if maps: results[gen]=np.array(maps); print(f'  {gen}: {len(maps)} ok, {fail} fail')
        np.save(cf, results); print(f'Saved: {cf}')
        return results

N_SAMPLES = 50
batch = BatchPhaseExtractor(CACHE_DIR)
phase_results = batch.extract(metadata_df, n_per_gen=N_SAMPLES)
print('\nSummary:')
for g,m in sorted(phase_results.items()): print(f'  {g:30s}: {m.shape}')

In [ ]:
# Cell 4: Mean Phase Maps
real_gens, fake_gens = [], []
for gen in phase_results:
    gdf = metadata_df[metadata_df['generator']==gen]
    if len(gdf)>0 and gdf['target'].mode().iloc[0]==0: real_gens.append(gen)
    else: fake_gens.append(gen)
if not real_gens:
    real_gens=[g for g in phase_results if 'real' in g.lower()]
    fake_gens=[g for g in phase_results if g not in real_gens]

mean_maps = {g: np.mean(m, axis=0) for g,m in phase_results.items()}
real_all = np.concatenate([phase_results[g] for g in real_gens if g in phase_results])
real_mean = np.mean(real_all, axis=0) if len(real_all)>0 else None
print(f'Real: {real_gens}\nFake: {fake_gens}')

disp = fake_gens[:8]
nc = min(len(disp)+1, 9)
fig,axes = plt.subplots(2, nc, figsize=(3*nc, 6))
fig.suptitle('Per-Generator Mean Phase Maps', fontweight='bold', y=1.02)
if real_mean is not None:
    axes[0,0].imshow(real_mean, cmap='twilight', vmin=0, vmax=1); axes[0,0].set_title('REAL',color='green',fontweight='bold'); axes[0,0].axis('off')
    axes[1,0].text(0.5,0.5,'REF',ha='center',va='center',transform=axes[1,0].transAxes); axes[1,0].axis('off')
for i,g in enumerate(disp):
    c=i+1
    if c>=nc: break
    axes[0,c].imshow(mean_maps[g],cmap='twilight',vmin=0,vmax=1); axes[0,c].set_title(g[:15],fontsize=8); axes[0,c].axis('off')
    if real_mean is not None:
        diff=np.abs(mean_maps[g]-real_mean)
        axes[1,c].imshow(diff,cmap='hot'); axes[1,c].set_title(f'|diff|',fontsize=8); axes[1,c].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Cell 5: KS Tests
real_samples = []
for g in real_gens:
    if g in phase_results:
        for m in phase_results[g]:
            flat=m.flatten(); idx=np.random.RandomState(42).choice(len(flat),size=min(5000,len(flat)),replace=False)
            real_samples.append(flat[idx])
real_dist = np.concatenate(real_samples)

ks_recs = []
for g in sorted(fake_gens):
    if g not in phase_results: continue
    fs = []
    for m in phase_results[g]:
        flat=m.flatten(); idx=np.random.RandomState(42).choice(len(flat),size=min(5000,len(flat)),replace=False)
        fs.append(flat[idx])
    fd = np.concatenate(fs)
    ks,pv = stats.ks_2samp(real_dist, fd)
    ks_recs.append({'generator':g,'n_images':len(phase_results[g]),'ks_stat':round(ks,6),'p_value':pv,'significant':'YES' if pv<0.05 else 'NO'})

ks_df = pd.DataFrame(ks_recs).sort_values('ks_stat',ascending=False)
print('KS Test Results:'); print(ks_df.to_string(index=False))
n_sig = (ks_df['significant']=='YES').sum()
print(f'\n{n_sig}/{len(ks_df)} generators significant (p<0.05)')
if n_sig>0: print('CONCLUSION: Phase-based detection IS viable.')

fig,(ax1,ax2) = plt.subplots(1,2,figsize=(14,max(4,len(ks_df)*0.5)))
fig.suptitle('KS Test: Real vs Fake Phase', fontweight='bold')
colors=['#e74c3c' if s=='YES' else '#95a5a6' for s in ks_df['significant']]
ax1.barh(ks_df['generator'],ks_df['ks_stat'],color=colors); ax1.set_xlabel('KS Stat'); ax1.invert_yaxis()
pv=ks_df['p_value'].replace(0,1e-300)
ax2.barh(ks_df['generator'],-np.log10(pv),color=colors); ax2.axvline(x=-np.log10(0.05),color='k',ls='--'); ax2.set_xlabel('-log10(p)'); ax2.invert_yaxis()
plt.tight_layout(); plt.show()